# HP 4192A User Guide

This notebook shows the HP 4192A API from a user point of view.

What it covers:

- open the instrument
- inspect state with `ping()` and `get()`
- change settings with `configure()`
- read measurements with `measure()`
- do a simple point-by-point frequency loop

This is not a bench test script. It is an example of normal API use.

In [ ]:
from pathlib import Path
import sys

# Find the repo root by walking upward until a folder named "source" exists.
cwd = Path.cwd().resolve()
for candidate in (cwd, *cwd.parents):
    if (candidate / "source").exists():
        REPO_ROOT = candidate
        break
else:
    raise RuntimeError("Could not find the repo root from the current notebook location.")

SOURCE_DIR = REPO_ROOT / "source"
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))

from instruments import HP4192A

## 1. Open The Instrument

Set the VISA resource string for your gateway and GPIB address, then open the instrument.

In [ ]:
RESOURCE = "TCPIP0::192.168.1.244::gpib0,5::INSTR"
TIMEOUT_MS = 5000

meter = HP4192A.open(RESOURCE, timeout_ms=TIMEOUT_MS)
meter

## 2. Inspect The Current Instrument State

Use `ping()` for a readable report and `get()` when you want one specific value.

In [ ]:
meter.ping()

print("frequency_hz:", meter.get("frequency_hz"))
print("bias_voltage_v:", meter.get("bias_voltage_v"))
print("osc_level_v:", meter.get("osc_level_v"))
print("display_a:", meter.get("display_a"))
print("display_b:", meter.get("display_b"))

## 3. Configure A Common Impedance Measurement

This example sets a simple impedance and phase measurement at 1 kHz.

Notice the style:

- high-level parameter names
- one `configure()` call
- no raw HP-IB codes in user code

In [ ]:
meter.configure(
    frequency_hz=1_000.0,
    osc_level_v=0.1,
    bias_enabled=False,
    trigger_mode="hold",
    measurement_mode="normal",
    zy_range="auto",
    circuit_mode="series",
    display_a="impedance",
    display_b="phase_deg",
)

meter.ping()

## 4. Read One Measurement

`measure()` returns a reading object with two numeric fields:

- `display_a`
- `display_b`

In [ ]:
reading = meter.measure()

print("DISPLAY A value:", reading.display_a)
print("DISPLAY B value:", reading.display_b)

## 5. Example: Capacitor Measurement

Connect a known capacitor, for example `1 uF`, then switch to capacitance and dissipation-factor mode.

This cell is a normal usage example. It is not checking correctness automatically. It just shows the API.

In [ ]:
meter.configure(
    frequency_hz=1_000.0,
    osc_level_v=0.1,
    bias_enabled=False,
    circuit_mode="parallel",
    display_a="capacitance",
    display_b="dissipation_factor",
)

reading = meter.measure()
print("capacitance:", reading.display_a)
print("dissipation factor:", reading.display_b)

## 6. Example: Resistor Measurement

Connect a known resistor and read impedance and phase.

In [ ]:
meter.configure(
    frequency_hz=1_000.0,
    osc_level_v=0.1,
    bias_enabled=False,
    circuit_mode="series",
    display_a="impedance",
    display_b="phase_deg",
)

reading = meter.measure()
print("impedance:", reading.display_a)
print("phase_deg:", reading.display_b)

## 7. Simple Point-By-Point Frequency Loop

This is not the full sweep application yet. It is only a simple example of how to call the current API repeatedly at different frequencies.

In [ ]:
frequencies_hz = [100, 1_000, 10_000, 100_000]
results = []

meter.configure(
    osc_level_v=0.1,
    bias_enabled=False,
    circuit_mode="series",
    display_a="impedance",
    display_b="phase_deg",
)

for frequency_hz in frequencies_hz:
    meter.configure(frequency_hz=frequency_hz)
    reading = meter.measure()
    results.append(
        {
            "frequency_hz": frequency_hz,
            "display_a": reading.display_a,
            "display_b": reading.display_b,
        }
    )

results

## 8. Close The Instrument

Run this when you are done.

In [ ]:
meter.close()